# QueryChat - `on_tool_request` Experimentation

In this notebook I'll be experimenting with the `on_tool_request` command to experiment intercepting LLM tool calls to validate, log, or transform them before execution. 

## Experiment 1: Log tool calls

This experiment is mostly to check what tool the LLM requests and what arguments it sends. For this experiment I will print both the **tool name** and the **arguments**. This could be useful to serve as a baseline and to unedrstand how querychat behaves **before** actually changing anything.

In [11]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0])) # add project root to path

from dotenv import load_dotenv
from querychat import QueryChat
from chatlas import ChatAnthropic
import os
import pandas as pd
from src.data import load_dashboard_data

In [12]:
# Experiment 1: Log tool requests with on_tool_request
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Load the same dataset used in the dashboard
df = load_dashboard_data()

# Reuse the same greeting and data description from app.py
ai_greeting = """
👋 Hi! I'm your AI burnout explorer.

Ask me questions about employee burnout, productivity, AI usage, workload, and work-life balance.

Examples:
- Show employees with high burnout risk
- Show employees with high AI usage and low productivity
- Sort employees by burnout risk from highest to lowest
- Which job roles have the highest burnout risk?
- Show employees with high manual work hours

You can also say Reset to clear the current AI filter/sort.
"""

ai_data_description = """
Employee-level workplace wellbeing and productivity dataset.

Each row represents one employee.

Columns:
- Employee_ID: unique identifier for each employee.
- job_role: employee job role/category.
- experience_years: years of experience.
- ai_tool_usage_hours_per_week: hours per week spent using AI tools.
- tasks_automated_percent: percent of tasks automated with AI/tools.
- manual_work_hours_per_week: hours per week spent on manual work.
- meeting_hours_per_week: hours per week spent in meetings.
- collaboration_hours_per_week: hours per week spent collaborating.
- focus_hours_per_day: average focus/deep work hours per day.
- deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
- burnout_risk_score: numeric burnout risk score.
- burnout_risk_level: burnout category label.
- productivity_score: numeric productivity score.
- work_life_balance_score: numeric work-life balance score.
- workload_score: derived workload metric combining manual work, meetings, and deadline pressure.
- workload_band: workload category (Low, Medium, High).
- ai_band: AI usage category (Low, Moderate, High).

This dataset can be used to analyze:
- Burnout risk by role or experience
- AI usage patterns across employees
- Links between productivity and burnout
- Work-life balance differences
- Manual work and deadline pressure patterns
- High-risk employee subgroups
"""

In [ ]:
# Store logs here
tool_logs = []
current_prompt = None

# Callback for on_tool_request
def log_tool_request(req):
    global current_prompt

    print("\n--- TOOL REQUEST ---")
    print("Prompt:", current_prompt)
    print("Tool name:", req.name)
    print("Arguments:", req.arguments)

    tool_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": req.name,
            "arguments": str(req.arguments),
        }
    )

# Create QueryChat client
client = ChatAnthropic(model="claude-sonnet-4-0")

# Create QueryChat object
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)

# Attach the callback to the client used by QueryChat
qc.client().on_tool_request(log_tool_request)

<function chatlas._callbacks.CallbackManager.add.<locals>._rm_callback() -> None>

In [14]:
test_prompts = [
    "Show employees with high burnout risk",
    "Show employees with high AI usage and low productivity",
    "Sort employees by burnout risk from highest to lowest",
    "Which job roles have the highest burnout risk?",
    "Show employees with high manual work hours"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = qc.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Show employees with high burnout risk


AttributeError: 'QueryChat' object has no attribute 'chat'

In [ ]:
log_df = pd.DataFrame(tool_logs)
log_df

## Experiment 2: Validate tool calls

In this experiment I add simple rules to check whether the request looks acceptable before the execution. Some validation rules I will try are:
- only allow expected tools like `query` or `update_dashboard`
- reject empty arguments
- reject overly board SQL queries like `SELECT *` without a filter
- reject requests that mention columns outside our dataset

This experiment is so that `on_tool_request` improves the reliability of the LLM.

## Experiment 3: Transform tool calls

In this experiment I will intercept the request to adjust it lightly before it runs. Some of the transformations I consider for this experiment are:
- cleaning the query title to standarize them
- normalize the column names
- improve query structure

This shows how `on_tool_request` can transform the tool requests before executions to something safer or more useful.